# 06 — Feature groups: SPY volume and VIX

*CRISP-DM phase 5 · code: `run_feature_groups.py`*

Each candidate group is added on top of the 13 main features and kept only if it improves out-of-sample PR-AUC by ≥ 0.01 for **both** logistic regression and LightGBM, without hurting onset PR-AUC (rule fixed before testing).

← previous: [`05_regime_models.ipynb`](05_regime_models.ipynb) · next: [`07_interpretation.ipynb`](07_interpretation.ipynb) →

In [1]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]   # project package + root scripts

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 100, "axes.titleweight": "bold", "font.size": 10})

In [2]:
def run_script(module, *args):
    """Run a project script's main() exactly as on the command line, then restore inline plots."""
    import importlib
    mod = importlib.import_module(module)
    old = sys.argv
    sys.argv = [f"{module}.py", *args]
    try:
        code = mod.main()
    finally:
        sys.argv = old
    get_ipython().run_line_magic("matplotlib", "inline")
    return code

## 1. Groups on a common 1993+ sample

In [3]:
run_script("run_feature_groups")

[18:23:52] Common sample: 1993-02-26 -> 2021, test years 2000-2021, models: logreg, lightgbm


Model is not converging.  Current: 16665.727885416116 is not greater than 16665.730563912242. Delta is -0.0026784961264638696


Model is not converging.  Current: 17565.230103420814 is not greater than 17565.279171995346. Delta is -0.049068574531702325


Model is not converging.  Current: 18397.279402551496 is not greater than 18397.336754539778. Delta is -0.05735198828188004


Model is not converging.  Current: 19251.58873011062 is not greater than 19251.646562819107. Delta is -0.05783270848769462


Model is not converging.  Current: 20219.74535312174 is not greater than 20219.918879115485. Delta is -0.17352599374498823


... (21 outputs left out)

[18:25:08]   base + vix             (16 features) done in 25s


[18:25:08] wrote reports/feature_groups.md and .csv



Mean over main targets:
                            pr_auc  onset_pr_auc
feature_set       method                        
base              lightgbm   0.527         0.138
                  logreg     0.441         0.109
base + spy_volume lightgbm   0.529         0.149
                  logreg     0.446         0.120
base + vix        lightgbm   0.545         0.156
                  logreg     0.461         0.152
spy_volume: DROP  gain={'lightgbm': 0.002, 'logreg': 0.005}  onset={'lightgbm': 0.01, 'logreg': 0.011}
vix: KEEP  gain={'lightgbm': 0.018, 'logreg': 0.02}  onset={'lightgbm': 0.018, 'logreg': 0.043}


0

In [4]:
display(Markdown((ROOT / "reports" / "feature_groups.md").read_text()))

# Feature group tests

Generated 2026-11-25 18:25 by `run_feature_groups.py`. Same rows for every feature set: training from 1993-02-26, walk-forward test years **2000–2021**, 21-day gap. Models: logistic regression and LightGBM (the two best in the main run). The 2022+ holdout is not used.

**Decision rule (fixed before testing):** keep a group only if mean PR-AUC over the 4 main targets (bear and stressed, 5 and 21 days) improves by **≥ 0.01** for **both** models, and mean onset PR-AUC drops by no more than 0.01 for either.

| Group | Features | Logistic regression: Δ PR-AUC | LightGBM: Δ PR-AUC | Δ onset PR-AUC (LR / LGBM) | Decision |
|---|---|---|---|---|---|
| spy_volume | `spy_vol_ratio_20d` | +0.005 | +0.002 | +0.011 / +0.010 | ❌ drop |
| vix | `vix`, `vix_rv_spread`, `vix_chg_5d` | +0.020 | +0.018 | +0.043 / +0.018 | ✅ keep |

## Main targets: PR-AUC by target

| Feature set | Model | bear 5d | bear 21d | stressed 5d | stressed 21d | Mean |
|---|---|---|---|---|---|---|
| base | Logistic regression | 0.143 | 0.113 | 0.938 | 0.569 | **0.441** |
| base | LightGBM | 0.237 | 0.233 | 0.952 | 0.685 | **0.527** |
| base + spy_volume | Logistic regression | 0.143 | 0.113 | 0.946 | 0.583 | **0.446** |
| base + spy_volume | LightGBM | 0.240 | 0.229 | 0.954 | 0.692 | **0.529** |
| base + vix | Logistic regression | 0.149 | 0.118 | 0.951 | 0.626 | **0.461** |
| base + vix | LightGBM | 0.276 | 0.266 | 0.954 | 0.683 | **0.545** |

Onset PR-AUC (main targets):

| Feature set | Model | bear 5d | bear 21d | stressed 5d | stressed 21d | Mean |
|---|---|---|---|---|---|---|
| base | Logistic regression | 0.077 | 0.033 | 0.171 | 0.155 | **0.109** |
| base | LightGBM | 0.005 | 0.022 | 0.263 | 0.263 | **0.138** |
| base + spy_volume | Logistic regression | 0.077 | 0.033 | 0.209 | 0.161 | **0.120** |
| base + spy_volume | LightGBM | 0.006 | 0.023 | 0.299 | 0.268 | **0.149** |
| base + vix | Logistic regression | 0.081 | 0.035 | 0.288 | 0.204 | **0.152** |
| base + vix | LightGBM | 0.006 | 0.024 | 0.289 | 0.305 | **0.156** |

## Robustness targets: PR-AUC by target

| Feature set | Model | bear_merged 5d | bear_merged 21d | stressed_fixed 5d | stressed_fixed 21d | Mean |
|---|---|---|---|---|---|---|
| base | Logistic regression | 0.167 | 0.132 | 0.900 | 0.626 | **0.456** |
| base | LightGBM | 0.265 | 0.279 | 0.921 | 0.612 | **0.519** |
| base + spy_volume | Logistic regression | 0.166 | 0.132 | 0.907 | 0.639 | **0.461** |
| base + spy_volume | LightGBM | 0.277 | 0.284 | 0.927 | 0.619 | **0.527** |
| base + vix | Logistic regression | 0.169 | 0.129 | 0.891 | 0.641 | **0.457** |
| base + vix | LightGBM | 0.275 | 0.282 | 0.907 | 0.620 | **0.521** |

Onset PR-AUC (robustness targets):

| Feature set | Model | bear_merged 5d | bear_merged 21d | stressed_fixed 5d | stressed_fixed 21d | Mean |
|---|---|---|---|---|---|---|
| base | Logistic regression | 0.003 | 0.011 | 0.225 | 0.256 | **0.124** |
| base | LightGBM | 0.003 | 0.012 | 0.333 | 0.270 | **0.155** |
| base + spy_volume | Logistic regression | 0.003 | 0.011 | 0.285 | 0.268 | **0.141** |
| base + spy_volume | LightGBM | 0.003 | 0.012 | 0.370 | 0.279 | **0.166** |
| base + vix | Logistic regression | 0.003 | 0.011 | 0.279 | 0.246 | **0.135** |
| base + vix | LightGBM | 0.003 | 0.013 | 0.308 | 0.279 | **0.151** |

## Year by year (`stressed` in 21 days)

Number of test years in which each set beat the base set on PR-AUC (years with both classes):

- Logistic regression: base + spy_volume: 14/16; base + vix: 14/16
- LightGBM: base + spy_volume: 9/16; base + vix: 9/16


## 2. VIX on the full 1962+ history (LightGBM, VIX missing before 1990)

In [5]:
run_script("run_feature_groups", "--vix-hybrid")

[18:25:08] VIX hybrid: LightGBM, train from 1962-01-01, test years 2000-2021


Model is not converging.  Current: 36231.48593276471 is not greater than 36231.5001094126. Delta is -0.014176647891872562


Model is not converging.  Current: 37144.78076848869 is not greater than 37144.78895812111. Delta is -0.008189632426365279


Model is not converging.  Current: 37983.84186529094 is not greater than 37983.865585435415. Delta is -0.023720144476101268


Model is not converging.  Current: 42398.32080946733 is not greater than 42398.333959955446. Delta is -0.013150488113751635


Model is not converging.  Current: 43300.468938946084 is not greater than 43300.501949524245. Delta is -0.033010578161338344


... (18 outputs left out)

[18:26:31]   main + vix, missing before 1990    done in 43s (5,536 test days)


[18:26:31] wrote reports/feature_groups_vix_hybrid.md and .csv


label                             bear        bear_merged        stressed        stressed_fixed       
horizon                             5      21          5      21       5      21             5      21
feature_set                                                                                           
main (13 features)               0.312  0.247        0.34  0.260    0.955  0.721          0.929  0.602
main + vix, missing before 1990  0.275  0.230        0.28  0.224    0.952  0.703          0.928  0.629

main-target gain -0.018, onset +0.001, robustness gain -0.018, years better 12/16 -> DROP


0

In [6]:
display(Markdown((ROOT / "reports" / "feature_groups_vix_hybrid.md").read_text()))

# VIX on the full history (LightGBM, VIX missing before 1990)

Generated 2026-11-25 18:26 by `run_feature_groups.py --vix-hybrid`. Both sets train LightGBM from 1962-01-01 on identical rows; the VIX features are empty (missing) before 1990 and LightGBM learns to handle that. Test years **2000–2021**. The 2022+ holdout is not used.

**Rule (same thresholds as the group tests, LightGBM only):** keep VIX if mean PR-AUC over the 4 main targets improves by ≥ 0.01 and mean onset PR-AUC drops by no more than 0.01.

| | Δ mean PR-AUC (main) | Δ mean onset PR-AUC (main) | Δ mean PR-AUC (robustness) | Years better (`stressed` 21d) | Decision |
|---|---|---|---|---|---|
| VIX added | -0.018 | +0.001 | -0.018 | 12/16 | ❌ drop |

## PR-AUC

| Feature set | bear 5d | bear 21d | bear_merged 5d | bear_merged 21d | stressed 5d | stressed 21d | stressed_fixed 5d | stressed_fixed 21d |
|---|---|---|---|---|---|---|---|---|
| main (13 features) | 0.312 | 0.247 | 0.340 | 0.260 | 0.955 | 0.721 | 0.929 | 0.602 |
| main + vix, missing before 1990 | 0.275 | 0.230 | 0.280 | 0.224 | 0.952 | 0.703 | 0.928 | 0.629 |

## Onset PR-AUC

| Feature set | bear 5d | bear 21d | bear_merged 5d | bear_merged 21d | stressed 5d | stressed 21d | stressed_fixed 5d | stressed_fixed 21d |
|---|---|---|---|---|---|---|---|---|
| main (13 features) | 0.005 | 0.020 | 0.003 | 0.012 | 0.313 | 0.302 | 0.330 | 0.206 |
| main + vix, missing before 1990 | 0.006 | 0.025 | 0.003 | 0.012 | 0.304 | 0.307 | 0.340 | 0.210 |

## Brier (lower is better)

| Feature set | bear 5d | bear 21d | bear_merged 5d | bear_merged 21d | stressed 5d | stressed 21d | stressed_fixed 5d | stressed_fixed 21d |
|---|---|---|---|---|---|---|---|---|
| main (13 features) | 0.108 | 0.117 | 0.130 | 0.142 | 0.039 | 0.113 | 0.031 | 0.080 |
| main + vix, missing before 1990 | 0.113 | 0.120 | 0.140 | 0.150 | 0.040 | 0.117 | 0.030 | 0.076 |



## 3. The 1990+ "modern" model with VIX, test 2000–2021

In [7]:
run_script("run_models", "--features", "modern")

[18:26:31] Walk-forward: features=modern (16), train from 1990-02-01, test 2000-2021, gap 21 days


[18:26:31]       fold 1/22: test 2000 (train 2,485 rows up to 1999-12-31)


[18:26:32]       fold 2/22: test 2001 (train 2,737 rows up to 2000-12-29)


[18:26:33]       fold 3/22: test 2002 (train 2,985 rows up to 2001-12-31)


[18:26:34]       fold 4/22: test 2003 (train 3,237 rows up to 2002-12-31)


[18:26:35]       fold 5/22: test 2004 (train 3,489 rows up to 2003-12-31)


... (22 outputs left out)

[18:27:06] Done in 35s; 265,728 predictions


[18:27:07] wrote reports/model_results_modern.md, .csv and docs/figures/model_timeline_modern.png



PR-AUC (main targets):
label         bear        stressed       
horizon         5      21       5      21
method                                   
persistence  0.135  0.129    0.816  0.526
ma200_rule   0.362  0.317      NaN    NaN
hmm          0.382  0.402    0.780  0.618
logreg       0.153  0.121    0.951  0.674
lightgbm     0.225  0.207    0.954  0.688
svm          0.290  0.294    0.738  0.583

Onset PR-AUC (main targets):
label         bear        stressed       
horizon         5      21       5      21
method                                   
persistence  0.006  0.024    0.035  0.102
ma200_rule   0.006  0.028      NaN    NaN
hmm          0.014  0.108    0.136  0.158
logreg       0.082  0.040    0.276  0.223
lightgbm     0.005  0.023    0.278  0.283
svm          0.011  0.045    0.145  0.148


0

In [8]:
a = pd.read_csv(ROOT / "reports" / "model_results_main.csv").assign(model="main (1962, no VIX)")
b = pd.read_csv(ROOT / "reports" / "model_results_modern.csv").assign(model="modern (1990, + VIX)")
both = pd.concat([a[a.method.isin(["logreg", "lightgbm"])], b[b.method.isin(["logreg", "lightgbm"])]])
both.pivot_table(index=["label", "horizon"], columns=["model", "method"], values="pr_auc").style.format("{:.3f}")

Note: the main model above is scored on 1980–2021 and the modern one on 2000–2021; the like-for-like comparison is in the report of section 1.

**Result.** SPY volume is dropped (below the bar). VIX passes on the common 1993+ sample but loses to the longer 1962 history in every full-history setup, so the main model stays VIX-free.